In [13]:
import pandas as pd, os
df = r'C:\Users\acer\Downloads\archive (2)'
print(os.listdir(d))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [14]:
o = pd.read_csv(os.path.join(d,"olist_orders_dataset.csv"), parse_dates=["order_purchase_timestamp","order_delivered_customer_date","order_estimated_delivery_date"])
i = pd.read_csv(os.path.join(d,"olist_order_items_dataset.csv"))
p = pd.read_csv(os.path.join(d,"olist_products_dataset.csv"))
t = pd.read_csv(os.path.join(d,"product_category_name_translation.csv"))
c = pd.read_csv(os.path.join(d,"olist_customers_dataset.csv"))
r = pd.read_csv(os.path.join(d,"olist_order_reviews_dataset.csv"))[["order_id","review_score"]].drop_duplicates("order_id")
pay = pd.read_csv(os.path.join(df,"olist_order_payments_dataset.csv")).groupby("order_id").agg(payment_type=("payment_type","first")).reset_index()

In [15]:
df = o.merge(i, on="order_id")
df = df.merge(p, on="product_id", how="left")
df = df.merge(t, on="product_category_name", how="left")
df = df.merge(c, on="customer_id")
df = df.merge(r, on="order_id", how="left")
df = df.merge(pay, on="order_id", how="left")
print(df.shape)

(112650, 29)


In [16]:
df = df[df["order_status"]=="delivered"]
df = df.dropna(subset=["order_delivered_customer_date"])
df["month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["is_late"] = (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(int)

In [17]:
df.to_csv("olist_clean.csv", index=False)
print(df.head())
print(os.getcwd())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08